# Clase 04 — Manipulación de Datos con Pandas

**Duplicados, `map`/`apply`, GroupBy y Pivot Tables, fechas y resampling — sobre un dataset real de rendimiento de jugadores.**

Antes de los 5 módulos nuevos hacemos un **Módulo 0 de repaso**: comandos de Pandas que ya usamos en la Clase 03 pero que no llegamos a explicar con el detalle que merecen (`.loc`/`.iloc`, filtros booleanos combinados, `value_counts`, `sort_values`, `rename`, `drop`, un `groupby` simple).

## Módulo 0 — Repaso: NumPy Relámpago y Comandos Clásicos de Pandas

### Sobre el dataset

Trabajamos con `fifa_world_cup_2026_player_performance.csv`: **54.600 filas**, cada una es la **aparición de un jugador en un partido** (no un jugador único — si un jugador disputó 5 partidos, aparece 5 veces). 75 columnas, agrupadas en tres familias: datos del jugador (`player_name`, `age`, `nationality`, `position`, `market_value_eur`...), datos del partido (`match_id`, `match_date`, `opponent_team`, `tournament_stage`...) y métricas de rendimiento en ese partido (`goals`, `assists`, `shots`, `pass_accuracy`, `minutes_played`, `player_rating`...).

Son 1.248 jugadores únicos, 48 equipos, 1.050 partidos, sin nulos ni filas duplicadas.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("fifa_world_cup_2026_player_performance.csv")

print(df.shape)     # (54600, 75)
df.info()

### 0.1 NumPy — repaso relámpago

Una columna de un DataFrame se puede convertir en un `ndarray` de NumPy con `.to_numpy()`. A partir de ahí, todas las operaciones son **vectorizadas**, sin `for`.

In [ ]:
valores_mercado = df["market_value_eur"].to_numpy()   # de Serie de Pandas a ndarray de NumPy
print(type(valores_mercado))

valores_en_millones = valores_mercado / 1_000_000       # vectorizado: sin for
print(f"Promedio: {valores_en_millones.mean():.2f}M | Máximo: {valores_en_millones.max():.2f}M")
print(f"Percentil 90: {np.percentile(valores_en_millones, 90):.2f}M")

### 0.2 Pandas — Series y DataFrame: lo esencial

`df["col"]` (un corchete) devuelve una **Serie**; `df[["col1", "col2"]]` (doble corchete) devuelve un **DataFrame**.

In [ ]:
nombres = df["player_name"]                          # un corchete -> Serie
ficha = df[["player_name", "team", "position"]]       # doble corchete -> DataFrame

print(type(nombres), type(ficha))
display(ficha.head(3))

### 0.3 Selección con `.loc[]` y `.iloc[]`

- **`.iloc[filas, columnas]`**: selección **por posición** (números enteros), sin importar cómo se llamen filas o columnas.
- **`.loc[filas, columnas]`**: selección **por etiqueta**, y se combina naturalmente con un filtro booleano en la parte de "filas".

In [ ]:
primeras_filas = df.iloc[0:5, 0:4]                 # posición: filas 0-4, columnas 0-3
print(primeras_filas)

goleadores = df.loc[df["goals"] > 2, ["player_name", "team", "goals"]]   # etiqueta: filtro + columnas por nombre
print(goleadores)

### 0.4 Filtrado booleano con múltiples condiciones

Sobre Series de Pandas se usan `&` (y), `|` (o) y `~` (no) — nunca `and`/`or`/`not` de Python puro — y cada condición individual va entre paréntesis.

`player_rating == 0` ocurre exactamente en las mismas filas donde `minutes_played == 0`: son jugadores convocados que no llegaron a jugar ese partido.

In [ ]:
jugaron = df[df["minutes_played"] > 0]                                              # excluye a los que no jugaron ese partido
delanteros_caros = df[(df["position"] == "Forward") & (df["market_value_eur"] > 50_000_000)]
no_finales = df[~(df["tournament_stage"] == "Final")]                               # ~ invierte la condición

print(f"Apariciones con minutos jugados: {len(jugaron)}")
print(f"Delanteros con valor > 50M€: {len(delanteros_caros)}")
print(f"Apariciones fuera de la Final: {len(no_finales)}")

### 0.5 Explorar categorías y ordenar: `value_counts()`, `unique()`, `nunique()`, `sort_values()`

- **`value_counts()`**: cuenta cuántas veces aparece cada valor distinto de una columna, de mayor a menor.
- **`nunique()`**: cuántos valores distintos hay (un solo número).
- **`sort_values("columna", ascending=False)`**: reordena todo el DataFrame según una columna.

In [ ]:
print(df["position"].value_counts())
print(df["team"].nunique())        # 48 equipos distintos

top_valor = df.sort_values("market_value_eur", ascending=False).head(5)
display(top_valor[["player_name", "team", "market_value_eur"]])

### 0.6 Preprocesamiento: nulos, `rename()` y `drop()`

`rename()` le pone otro nombre a una columna sin tocar los datos; `drop(columns=[...])` saca columnas completas (para filas se usa `index=[...]`).

In [ ]:
print(df.isnull().sum().sum())    # 0 -> este dataset viene sin nulos

df = df.rename(columns={"goals": "goles", "assists": "asistencias"})   # renombrar para trabajar en español
df = df.drop(columns=["jersey_number"])                                 # sacar una columna que no vamos a usar

print(df.columns[:5].tolist())

### 0.7 Agregación básica con `groupby`

El caso más simple: una columna para agrupar y una sola métrica. El `groupby` con múltiples agregaciones a la vez y las `pivot_table` se ven más adelante en esta clase.

In [ ]:
valor_por_posicion = df.groupby("position")["market_value_eur"].mean().sort_values(ascending=False)
print((valor_por_posicion / 1_000_000).round(2))

### 0.8 Sinergia NumPy + Pandas

`np.where(condición, si_true, si_false)` es un `if/else` vectorizado, aplicado a toda una columna de una sola vez — sin `for` y sin `.apply()` (que recién se introduce en el próximo módulo).

In [ ]:
df["tuvo_gol"] = np.where(df["goles"] > 0, "Sí", "No")   # if/else vectorizado, sin for ni apply
print(df["tuvo_gol"].value_counts())